# MK vs RelCal pairwise angle comparison
This notebook prints the pairwise MK angle differences $\alpha_i-\alpha_j$ from the MK chain and the corresponding relative-calibration estimate $\Delta\alpha_{ij}$ from the antisymmetric cross-spectrum method.
The two quantities are compared in the same gauge, so the reference map only fixes the zero point. The table also reports the signed disagreement in units of the combined 1$\sigma$ uncertainty.

In [1]:
import os
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[variable] = '1'

%cd /global/homes/l/lonappan/workspace/cosmic_birefringence
%matplotlib inline
import numpy as np
from cosmic_bire import CBlike
from cosmic_bire.relcal import RelCal

config = 'configs/planck_hfi.yml'
mask = '30'
ref = '353B'

rc = RelCal(config, mask=mask)
labels = [rc.map_label(k) for k in range(rc.nmaps)]
print('maps:', labels)

lh = CBlike(config)
_ = lh.getdist_samples()
samples = lh.samples
mk_alpha = samples[:, 1:1 + rc.nmaps]
mk_mean = mk_alpha.mean(axis=0)
mk_cov = np.cov(mk_alpha.T)

pair_fits = rc.fit_all_pairs()

/global/u2/l/lonappan/workspace/cosmic_birefringence
maps: ['100A', '143A', '217A', '353A', '100B', '143B', '217B', '353B']
Loaded 218368 samples directly from /global/homes/l/lonappan/pscratch/CBDATA/chains/planck_hfi_mask_0.h5
Removed no burn in


In [2]:
def mk_pair_difference(a, b):
    ia = labels.index(a)
    ib = labels.index(b)
    value = mk_mean[ia] - mk_mean[ib]
    sigma = np.sqrt(mk_cov[ia, ia] + mk_cov[ib, ib] - 2.0 * mk_cov[ia, ib])
    return value, sigma

print(f'{"pair":12s} {"MK alpha_i-alpha_j":>22s} {"RelCal delta_alpha":>22s} {"sigma diff":>12s}')
for (a, b), (delta_alpha, delta_sigma) in pair_fits.items():
    mk_value, mk_sigma = mk_pair_difference(a, b)
    sigma_diff = np.nan if np.isclose(mk_sigma**2 + delta_sigma**2, 0.0) else (mk_value - delta_alpha) / np.sqrt(mk_sigma**2 + delta_sigma**2)
    sigma_text = '   n/a' if np.isnan(sigma_diff) else f'{sigma_diff:+9.2f}\u03c3'
    print(f'{a}-{b:8s} {mk_value:+10.4f} +- {mk_sigma:6.4f}   {delta_alpha:+10.4f} +- {delta_sigma:6.4f} {sigma_text:>12s}')

pair             MK alpha_i-alpha_j     RelCal delta_alpha   sigma diff
100A-143A        -0.3423 +- 0.0890      -0.3361 +- 0.1810       -0.03σ
100A-217A        -0.2559 +- 0.0859      -0.3147 +- 0.2253       +0.24σ
100A-353A        -0.1295 +- 0.0868      +0.4476 +- 0.3908       -1.44σ
100A-100B        +0.1052 +- 0.1022      +0.3730 +- 0.2130       -1.13σ
100A-143B        -0.4623 +- 0.0881      -0.4300 +- 0.1745       -0.16σ
100A-217B        -0.2360 +- 0.0869      -0.1470 +- 0.2251       -0.37σ
100A-353B        -0.0922 +- 0.0876      -0.2351 +- 0.4775       +0.29σ
143A-217A        +0.0864 +- 0.0529      +0.4316 +- 0.1592       -2.06σ
143A-353A        +0.2128 +- 0.0524      +0.1537 +- 0.3130       +0.19σ
143A-100B        +0.4474 +- 0.0819      +0.4819 +- 0.1628       -0.19σ
143A-143B        -0.1200 +- 0.0594      +0.0199 +- 0.1269       -1.00σ
143A-217B        +0.1062 +- 0.0531      +0.0345 +- 0.1587       +0.43σ
143A-353B        +0.2500 +- 0.0542      +0.3988 +- 0.3231       -0.45σ
217A-

In [3]:
# Compare RelCal pairwise delta_alpha between mask 0 and mask 30
rc0 = RelCal(config, mask='0')
rc30 = RelCal(config, mask='30')
pairs0 = rc0.fit_all_pairs()
pairs30 = rc30.fit_all_pairs()

print(f'{"pair":12s} {"mask0 delta_alpha":>18s} {"mask30 delta_alpha":>18s} {"diff":>12s} {"sigma shift":>14s}')
for pair in pairs0:
    d0, s0 = pairs0[pair]
    d30, s30 = pairs30[pair]
    diff = d0 - d30
    sigma_shift = np.nan if np.isclose(s0**2 + s30**2, 0.0) else diff / np.sqrt(s0**2 + s30**2)
    sigma_text = '   n/a' if np.isnan(sigma_shift) else f'{sigma_shift:+9.2f}\u03c3'
    print(f'{pair[0]}x{pair[1]:8s} {d0:+10.4f} +- {s0:6.4f}   {d30:+10.4f} +- {s30:6.4f} {diff:+10.4f} {sigma_text:>14s}')

pair          mask0 delta_alpha mask30 delta_alpha         diff    sigma shift
100Ax143A        -0.3066 +- 0.1199      -0.3361 +- 0.1810    +0.0295         +0.14σ
100Ax217A        -0.3552 +- 0.1307      -0.3147 +- 0.2253    -0.0405         -0.16σ
100Ax353A        -0.0880 +- 0.1688      +0.4476 +- 0.3908    -0.5356         -1.26σ
100Ax100B        +0.1051 +- 0.1487      +0.3730 +- 0.2130    -0.2679         -1.03σ
100Ax143B        -0.4522 +- 0.1168      -0.4300 +- 0.1745    -0.0222         -0.11σ
100Ax217B        -0.2327 +- 0.1303      -0.1470 +- 0.2251    -0.0856         -0.33σ
100Ax353B        -0.2231 +- 0.1741      -0.2351 +- 0.4775    +0.0120         +0.02σ
143Ax217A        +0.1359 +- 0.0650      +0.4316 +- 0.1592    -0.2957         -1.72σ
143Ax353A        +0.2019 +- 0.0704      +0.1537 +- 0.3130    +0.0482         +0.15σ
143Ax100B        +0.4835 +- 0.1088      +0.4819 +- 0.1628    +0.0016         +0.01σ
143Ax143B        -0.1002 +- 0.0712      +0.0199 +- 0.1269    -0.1201         -0.8

In [4]:
# Optional compact summary table for the chosen gauge reference
ref_idx = labels.index(ref)
mk_rel = mk_mean - mk_mean[ref_idx]
mk_rel_err = np.sqrt(np.abs(np.diag(mk_cov) + mk_cov[ref_idx, ref_idx] - 2.0 * mk_cov[:, ref_idx]))
rel_ref, rel_cov, _ = rc.fit_network(ref=ref)
rel_err = np.sqrt(np.abs(np.diag(rel_cov)))

print(f'Reference map: {ref}')
print(f'{"map":5s} {"MK alpha- alpha_ref":>22s} {"RelCal alpha- alpha_ref":>26s}')
for k, label in enumerate(labels):
    print(f'{label:5s} {mk_rel[k]:+10.4f} +- {mk_rel_err[k]:6.4f}   {rel_ref[k]:+10.4f} +- {rel_err[k]:6.4f}')

Reference map: 353B
map      MK alpha- alpha_ref    RelCal alpha- alpha_ref
100A     -0.0922 +- 0.0876      -0.0867 +- 0.2011
143A     +0.2500 +- 0.0542      +0.2561 +- 0.1814
217A     +0.1636 +- 0.0325      +0.0521 +- 0.1674
353A     +0.0372 +- 0.0232      -0.0653 +- 0.1449
100B     -0.1974 +- 0.0809      -0.2285 +- 0.1942
143B     +0.3700 +- 0.0527      +0.2751 +- 0.1792
217B     +0.1438 +- 0.0323      +0.0083 +- 0.1675
353B     +0.0000 +- 0.0000      +0.0000 +- 0.0831
